# EVS Life Satisfaction: v0.1.0 quickstart

This example uses **invented profiles**, not survey records. Predictions describe historical EVS-style contexts; changing an input does not estimate an intervention effect. No raw dataset is needed.

In [1]:
from pathlib import Path
import pandas as pd
from evs_model.cli import load_bundle, predict_frame

ROOT = Path.cwd()
if not (ROOT / "models").is_dir():
    ROOT = ROOT.parent
bundle = load_bundle(ROOT / "models/evs_v0_1_0.joblib")
print("Model:", bundle["model_name"], "| training respondents:", bundle["training_rows"])
print("Supported countries:", ", ".join(bundle["countries"]))

Model: boosting | training respondents: 34740
Supported countries: AL, AM, AZ, BA, BG, BY, CH, CZ, DE, DK, EE, ES, FI, GE, HR, HU, IS, IT, LV, ME, NL, NO, PL, PT, RO, RU, SE, SI, SK, UA


## Synthetic inputs
Use the official codebook: X001 sex, X003 age (82 means 82+), X025R education, X028 employment, X047E_EVS5 income decile, A009 health, A165 trust, A173 control, cntry_AN country. Blank and -1 through -5 inputs are handled as missing. The outcome must not be supplied.

In [2]:
profiles = pd.read_csv(ROOT / "examples/respondents.csv")
profiles

,X001,X003,X025R,X028,X047E_EVS5,A009,A165,A173,cntry_AN
0,1,35,3,1,6.0,2,1,7,CZ
1,2,68,2,4,3.0,3,2,5,SK
2,2,82,2,4,NaN,2,1,8,CZ


In [3]:
predictions = predict_frame(bundle, profiles)
predictions.round(4)

,predicted_score,probability_high_7_10,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10
0,8,0.8475,0.0032,0.0017,0.0079,0.0085,0.0379,0.0933,0.2759,0.3593,0.1746,0.0378
1,6,0.4484,0.0090,0.0088,0.0321,0.0659,0.3153,0.1205,0.2201,0.1436,0.0437,0.0409
2,9,0.9447,0.0020,0.0011,0.0023,0.0025,0.0158,0.0316,0.0715,0.3212,0.3096,0.2423


In [4]:
probabilities = predictions[[f"p_{i}" for i in range(1, 11)]]
assert probabilities.ge(0).all().all()
assert probabilities.sum(axis=1).sub(1).abs().max() < 1e-10
print("All class probabilities are valid.")

All class probabilities are valid.


## Batch command
From the repository root:
```bash
evs-predict --input examples/respondents.csv --output predictions.csv
```
The artifact includes its fitted preprocessing. For a different working directory pass `--model /path/to/models/evs_v0_1_0.joblib`. See the metadata and model card for versions and scope. Final prediction-test evaluation belongs to Assure.

In [5]:
try:
    predict_frame(bundle, profiles.assign(A170=8))
except ValueError as error:
    print("Expected input rejection:", error)

Expected input rejection: Remove target/administrative columns: A170
